# Analista de Experiencia del Cliente: 29/06/2026
## Pregunta de negocio

**¿Cuál es la puntuación media dada por los usuarios a los alojamientos
turísticos y qué porcentaje de alojamientos tienen una evaluación
general superior a 80 en cada ciudad?**

Este notebook corresponde al perfil de **Analista de Experiencia del
Cliente**. Utiliza el dataset común generado después de las fases de
Data Cleaning, Data Transformation y Data Reduction.

## Desglose y preparación de la pregunta-respuesta

### Variables principales

- `apartment_id`: identificador del alojamiento.
- `city`: ciudad del alojamiento.
- `review_scores_rating`: valoración general, en escala de 0 a 100.

### Métricas principales

- Puntuación media global.
- Puntuación media por ciudad.
- Número de alojamientos con valoración válida.
- Número y porcentaje de alojamientos con puntuación superior a 80
  por ciudad.

### Análisis complementario

Si las variables están disponibles en el dataset reducido, también se
compararán los distintos ítems de satisfacción en escala de 0 a 10.

Los valores nulos de las puntuaciones no se imputan. Para cada KPI se
utilizan únicamente los alojamientos que disponen de una valoración
válida.

## Relación con el trabajo previo del equipo

Este análisis parte de las decisiones tomadas en Data Understanding y
EDA:

- La valoración general se obtiene de `review_scores_rating`.
- La escala esperada de la valoración general es de 0 a 100.
- Los demás ítems de satisfacción deben estar en escala de 0 a 10.
- Para el estado actual se utiliza el último registro de cada
  `apartment_id`, decisión aplicada antes de este notebook.
- Los alojamientos sin reseñas se mantienen en el dataset común, pero
  no participan en el cálculo de los KPI de satisfacción.

Cuando se reciba el dataset reducido definitivo, solo será necesario
actualizar la ruta del archivo y ejecutar **Run All**.

## 1. Importación de librerías

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

## 2. Carga del dataset reducido

La ruta es provisional. Debe sustituirse por la ubicación definitiva
del archivo generado en la fase de Data Reduction.

In [ ]:
file_path = Path(
    "../Data/tourist_accommodation_reduced.csv"
)

if not file_path.exists():
    raise FileNotFoundError(
        "Actualiza 'file_path' con la ruta del dataset reducido final."
    )

df = pd.read_csv(file_path)

print(f"Número de registros: {df.shape[0]:,}")
print(f"Número de columnas: {df.shape[1]}")
df.head()

## 3. Validación del dataset de entrada

Esta sección no repite el Data Cleaning. Solo comprueba que el archivo
recibido contiene las variables y escalas necesarias para el análisis.

In [ ]:
required_columns = [
    "apartment_id",
    "city",
    "review_scores_rating",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Faltan columnas necesarias: "
        + ", ".join(missing_columns)
    )

print("Las columnas principales están disponibles.")

In [ ]:
df[required_columns].info()

In [ ]:
rating_summary = df["review_scores_rating"].describe()
rating_summary

In [ ]:
invalid_rating_mask = (
    df["review_scores_rating"].notna()
    & ~df["review_scores_rating"].between(0, 100)
)

invalid_rating_count = invalid_rating_mask.sum()

print(
    "Valores de review_scores_rating fuera del rango 0-100:",
    invalid_rating_count,
)

if invalid_rating_count > 0:
    display(
        df.loc[
            invalid_rating_mask,
            required_columns,
        ].head()
    )

## 4. Preparación de la base analítica

Para responder a la pregunta se seleccionan únicamente los registros
con ciudad y valoración general válidas. No se rellenan puntuaciones
nulas, ya que hacerlo introduciría valoraciones artificiales.

In [ ]:
df_customer = (
    df.loc[
        :,
        required_columns,
    ]
    .dropna(
        subset=[
            "city",
            "review_scores_rating",
        ]
    )
    .copy()
)

print(
    "Alojamientos con valoración válida:",
    df_customer["apartment_id"].nunique(),
)
print(
    "Registros utilizados en el análisis:",
    len(df_customer),
)

## 5. Puntuación media global

In [ ]:
global_average_rating = round(
    df_customer["review_scores_rating"].mean(),
    2,
)

print(
    "Puntuación media global:",
    global_average_rating,
)

## 6. Puntuación media y cobertura por ciudad

In [ ]:
total_by_city = (
    df.groupby("city")["apartment_id"]
    .nunique()
    .rename("total_accommodations")
)

rated_by_city = (
    df_customer.groupby("city")["apartment_id"]
    .nunique()
    .rename("rated_accommodations")
)

average_by_city = (
    df_customer.groupby("city")["review_scores_rating"]
    .mean()
    .round(2)
    .rename("average_rating")
)

city_rating_summary = pd.concat(
    [
        total_by_city,
        rated_by_city,
        average_by_city,
    ],
    axis=1,
).reset_index()

city_rating_summary["rating_coverage_pct"] = (
    city_rating_summary["rated_accommodations"]
    / city_rating_summary["total_accommodations"]
    * 100
).round(2)

city_rating_summary = city_rating_summary.sort_values(
    "average_rating",
    ascending=False,
)

city_rating_summary

## 7. Alojamientos con puntuación superior a 80

El denominador del porcentaje está formado únicamente por los
alojamientos que tienen una valoración general válida.

In [ ]:
df_customer["rating_over_80"] = (
    df_customer["review_scores_rating"] > 80
)

over_80_by_city = (
    df_customer.groupby("city")
    .agg(
        accommodations_over_80=(
            "rating_over_80",
            "sum",
        ),
        valid_ratings=(
            "rating_over_80",
            "count",
        ),
    )
    .reset_index()
)

over_80_by_city["percentage_over_80"] = (
    over_80_by_city["accommodations_over_80"]
    / over_80_by_city["valid_ratings"]
    * 100
).round(2)

over_80_by_city = over_80_by_city.sort_values(
    "percentage_over_80",
    ascending=False,
)

over_80_by_city

## 8. Tabla final de respuesta a la pregunta de negocio

In [ ]:
customer_experience_summary = (
    city_rating_summary
    .merge(
        over_80_by_city,
        on="city",
        how="left",
    )
    .sort_values(
        "average_rating",
        ascending=False,
    )
)

customer_experience_summary

## 9. Visualización de la puntuación media por ciudad

In [ ]:
plot_average = city_rating_summary.sort_values(
    "average_rating",
    ascending=False,
)

ax = plot_average.plot(
    x="city",
    y="average_rating",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)

ax.set_title(
    "Puntuación media de los alojamientos por ciudad"
)
ax.set_xlabel("Ciudad")
ax.set_ylabel("Puntuación media (0-100)")
ax.tick_params(
    axis="x",
    rotation=45,
)

plt.tight_layout()
plt.show()

## 10. Visualización del porcentaje superior a 80

In [ ]:
plot_over_80 = over_80_by_city.sort_values(
    "percentage_over_80",
    ascending=False,
)

ax = plot_over_80.plot(
    x="city",
    y="percentage_over_80",
    kind="bar",
    figsize=(10, 5),
    legend=False,
)

ax.set_title(
    "Alojamientos con puntuación superior a 80 por ciudad"
)
ax.set_xlabel("Ciudad")
ax.set_ylabel("Porcentaje de alojamientos (%)")
ax.tick_params(
    axis="x",
    rotation=45,
)

plt.tight_layout()
plt.show()

## 11. Análisis complementario de los ítems de satisfacción

Este bloque se ejecutará si las columnas correspondientes se han
conservado en el dataset reducido.

In [ ]:
satisfaction_columns = [
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
]

available_satisfaction_columns = [
    column
    for column in satisfaction_columns
    if column in df.columns
]

if available_satisfaction_columns:
    item_summary = (
        df[available_satisfaction_columns]
        .mean()
        .round(2)
        .sort_values(ascending=False)
        .rename("average_score")
        .reset_index(names="satisfaction_item")
    )

    display(item_summary)

    best_item = item_summary.iloc[0]
    worst_item = item_summary.iloc[-1]

    print(
        "Ítem mejor valorado:",
        best_item["satisfaction_item"],
        "-",
        best_item["average_score"],
    )
    print(
        "Ítem peor valorado:",
        worst_item["satisfaction_item"],
        "-",
        worst_item["average_score"],
    )
else:
    print(
        "Las columnas de satisfacción complementaria "
        "no están disponibles en el dataset reducido."
    )

In [ ]:
if available_satisfaction_columns:
    ax = item_summary.sort_values(
        "average_score",
        ascending=True,
    ).plot(
        x="satisfaction_item",
        y="average_score",
        kind="barh",
        figsize=(9, 5),
        legend=False,
    )

    ax.set_title(
        "Puntuación media por ítem de satisfacción"
    )
    ax.set_xlabel("Puntuación media (0-10)")
    ax.set_ylabel("Ítem de satisfacción")

    plt.tight_layout()
    plt.show()

## 12. Indicadores principales automáticos

Este bloque identifica las ciudades con los valores más altos y más
bajos después de ejecutar el notebook con el dataset definitivo.

In [ ]:
highest_average_city = (
    city_rating_summary
    .sort_values(
        "average_rating",
        ascending=False,
    )
    .iloc[0]
)

lowest_average_city = (
    city_rating_summary
    .sort_values(
        "average_rating",
        ascending=True,
    )
    .iloc[0]
)

highest_over_80_city = (
    over_80_by_city
    .sort_values(
        "percentage_over_80",
        ascending=False,
    )
    .iloc[0]
)

lowest_over_80_city = (
    over_80_by_city
    .sort_values(
        "percentage_over_80",
        ascending=True,
    )
    .iloc[0]
)

print(
    "Mayor puntuación media:",
    highest_average_city["city"],
    "-",
    highest_average_city["average_rating"],
)
print(
    "Menor puntuación media:",
    lowest_average_city["city"],
    "-",
    lowest_average_city["average_rating"],
)
print(
    "Mayor porcentaje superior a 80:",
    highest_over_80_city["city"],
    "-",
    highest_over_80_city["percentage_over_80"],
)
print(
    "Menor porcentaje superior a 80:",
    lowest_over_80_city["city"],
    "-",
    lowest_over_80_city["percentage_over_80"],
)

## 13. Conclusiones

> **Pendiente de completar después de ejecutar el notebook con el
> dataset final limpio, transformado y reducido.**

La conclusión final debe responder de manera directa:

1. ¿Cuál es la puntuación media global?
2. ¿Qué ciudad tiene la puntuación media más alta y más baja?
3. ¿Qué porcentaje de alojamientos supera 80 en cada ciudad?
4. ¿Qué ciudades presentan mayor y menor cobertura de valoraciones?
5. ¿Cuál es el ítem mejor valorado y cuál necesita más mejora?

Los resultados de una versión previa del dataset pueden utilizarse
como referencia preliminar, pero las conclusiones definitivas deben
actualizarse después del último **Run All**.

## 14. Exportación opcional de resultados

La tabla final puede guardarse para utilizarla en el dashboard o en
la presentación del Sprint.

In [ ]:
output_path = Path(
    "../Data/customer_experience_summary.csv"
)

# Descomentar cuando la ruta de salida esté confirmada.
# customer_experience_summary.to_csv(
#     output_path,
#     index=False,
# )

print(
    "Ruta prevista para la exportación:",
    output_path,
)